In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "OSQ_H.xpt"
csv_file = DATA_DIR / "OSQ_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 OSQ_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",

    "OSQ010A",
    "OSQ010B",
    "OSQ010C",

    "OSQ020A",
    "OSQ020B",
    "OSQ020C",

    "OSD030AA",
    "OSQ040AA",
    "OSD050AA",

    "OSD030AB",
    "OSQ040AB",
    "OSD050AB",

    "OSD030AC",
    "OSQ040AC",
    "OSD050AC",

    "OSD030BA",
    "OSQ040BA",
    "OSD050BA",

    "OSD030BB",
    "OSQ040BB",
    "OSD050BB",

    "OSD030BC",
    "OSQ040BC",
    "OSD050BC",

    "OSD030BD",
    "OSQ040BD",
    "OSD050BD",

    "OSD030BE",
    "OSQ040BE",

    "OSD030BF",
    "OSQ040BF",

    "OSD030BG",
    "OSQ040BG",

    "OSD030BH",
    "OSQ040BH",

    "OSD030BI",
    "OSQ040BI",

    "OSD030BJ",
    "OSQ040BJ",

    "OSD030CA",
    "OSQ040CA",
    "OSD050CA",

    "OSD030CB",
    "OSQ040CB",
    "OSD050CB",

    "OSD030CC",
    "OSQ040CC",

    "OSQ080",

    "OSQ090A",
    "OSQ100A",
    "OSD110A",
    "OSQ120A",

    "OSQ090B",
    "OSQ100B",
    "OSD110B",
    "OSQ120B",

    "OSQ090C",
    "OSQ100C",
    "OSD110C",
    "OSQ120C",

    "OSQ090D",
    "OSQ100D",
    "OSD110D",
    "OSQ120D",

    "OSQ090E",
    "OSQ100E",
    "OSD110E",
    "OSQ120E",

    "OSQ090F",
    "OSQ120F",

    "OSQ090G",
    "OSQ100G",
    "OSD110G",
    "OSQ120G",

    "OSQ090H",
    "OSQ120H",

    "OSQ060",
    "OSQ072",

    "OSQ130",
    "OSQ140Q",
    "OSQ140U",

    "OSQ150",
    "OSQ160A",
    "OSQ160B",

    "OSQ170",
    "OSQ180",
    "OSQ190",

    "OSQ200",
    "OSQ210",
    "OSQ220"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "OSQ_H column names/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 3815:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 91:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 3,815 rows x 91 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached OSQ_H:
#
#     genuine decimal fields = NONE
#     string fields          = NONE
#
# Every released variable is:
#
#     SEQN
#     response code
#     age
#     fracture count
#     duration quantity
#     unit code
#
# and every populated value is whole-number.
#
# Therefore all 91 fields can safely become Int64.
#
#
# IMPORTANT:
#
# This applies ONLY to OSQ_H.
#
# It does NOT change the general NHANES rule:
#
# Genuine Dietary / Lab / Examination / sample-weight
# decimals remain exactly AS IS.
#

DECIMAL_COLUMNS = []

STRING_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 91:

    raise ValueError(
        "Expected exactly 91 whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "String fields:",
    STRING_COLUMNS
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT artifact:
#
#     5.397605346934028e-79
#
# Attached OSQ_H contains NONE.
#
# Therefore NO tiny -> zero replacement is appropriate.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 91 OSQ_H variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in OSQ_H."
    )


if true_zero_total > 0:

    print(
        "\nOrdinary zeros by variable:"
    )

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected ordinary zeros "
        "found in attached OSQ_H."
    )


print(
    "PASS: Attached OSQ_H contains "
    "0 tiny artifacts and 0 ordinary zeros."
)


# ============================================================
# 6. CONFIRM NO ZERO CORRECTION IS NEEDED
# ============================================================

#
# Unlike files where CDC documents valid zero values,
# OSQ_H has no decoded tiny-zero artifact in this
# attachment.
#
# Therefore:
#
#     DO NOT replace anything with zero.
#

expected_df = source_df.copy()


print(
    "\nPASS: No numeric correction required."
)


# ============================================================
# 7. CHECK FOR GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# MUST run before Int64 conversion.
#
# np.round() is used ONLY to determine whether a
# value is fractional.
#
# It does NOT alter the dataframe.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(
            values
        ),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{fractional_count:,} fractional values"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal fields unexpectedly "
        "found in OSQ_H. They will NOT be rounded:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: No genuine fractional variables "
    "across all 91 fields."
)


# ============================================================
# 8. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(
        seqn_values
    ),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83729:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicates."
    )


print(
    "PASS: SEQN 73557-83729 validated."
)


# ============================================================
# 9. CONVERT ALL PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# All 91 variables have already been proven to contain
# whole-number values only.
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(
                values
            ),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 91 OSQ_H fields converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 10. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches expected "
        "CDC release frequencies."
    )


# ============================================================
# 11. HELPER - RANGE/SPECIAL CODE CHECK
# ============================================================

def check_range_special(
    column,
    low,
    high,
    expected_range_count,
    expected_missing,
    special_counts=None
):

    if special_counts is None:

        special_counts = {}


    s = df[column]


    actual_range_count = int(
        s.between(
            low,
            high
        )
        .sum()
    )


    actual_missing = int(
        s.isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        f"Range {low}-{high}:",
        f"{actual_range_count:,}"
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_range_count != expected_range_count:

        raise ValueError(
            f"{column}: range count differs.\n"
            f"Expected: {expected_range_count:,}\n"
            f"Actual:   {actual_range_count:,}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column}: missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    for code, expected_count in special_counts.items():

        actual_count = int(
            s.eq(code)
            .sum()
        )


        print(
            f"Code {code}:",
            actual_count
        )


        if actual_count != expected_count:

            raise ValueError(
                f"{column}: code {code} count differs.\n"
                f"Expected: {expected_count:,}\n"
                f"Actual:   {actual_count:,}"
            )


    print(
        f"PASS: {column} range/special "
        "codes validated."
    )


# ============================================================
# 12. CDC CHECK - OSQ010A
# ============================================================

#
# Ever broken/fractured hip.
#
# CDC:
#
# 1 Yes         83
# 2 No        3731
# 9 Don't know  1
#

check_frequency(
    "OSQ010A",
    {
        1: 83,
        2: 3731,
        9: 1
    },
    0
)


# ============================================================
# 13. CDC CHECK - OSQ010B
# ============================================================

#
# Ever broken/fractured wrist.
#

check_frequency(
    "OSQ010B",
    {
        1: 316,
        2: 3496,
        9: 3
    },
    0
)


# ============================================================
# 14. CDC CHECK - OSQ010C
# ============================================================

#
# Ever broken/fractured spine.
#

check_frequency(
    "OSQ010C",
    {
        1: 77,
        2: 3734,
        9: 4
    },
    0
)


# ============================================================
# 15. FRACTURE COUNT ROUTING - HIP
# ============================================================

#
# OSQ020A populated exactly when:
#
#     OSQ010A = 1
#

hip_yes = (
    df["OSQ010A"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


hip_count_present = (
    df["OSQ020A"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


hip_route_mismatch = int(
    np.sum(
        hip_yes
        !=
        hip_count_present
    )
)


if hip_route_mismatch != 0:

    raise ValueError(
        "OSQ020A routing does not match OSQ010A=1."
    )


print(
    "PASS: Hip-fracture count routing validated."
)


# ============================================================
# 16. CDC CHECK - OSQ020A
# ============================================================

check_frequency(
    "OSQ020A",
    {
        1: 71,
        2: 9,
        3: 2,
        9999: 1
    },
    3732
)


# ============================================================
# 17. FRACTURE COUNT ROUTING - WRIST
# ============================================================

wrist_yes = (
    df["OSQ010B"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


wrist_count_present = (
    df["OSQ020B"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        wrist_yes
        !=
        wrist_count_present
    )
) != 0:

    raise ValueError(
        "OSQ020B routing does not match OSQ010B=1."
    )


print(
    "PASS: Wrist-fracture count routing validated."
)


# ============================================================
# 18. CDC CHECK - OSQ020B
# ============================================================

check_frequency(
    "OSQ020B",
    {
        1: 267,
        2: 35,
        3: 9,
        4: 3,
        6: 1,
        10: 1
    },
    3499
)


# ============================================================
# 19. FRACTURE COUNT ROUTING - SPINE
# ============================================================

spine_yes = (
    df["OSQ010C"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


spine_count_present = (
    df["OSQ020C"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        spine_yes
        !=
        spine_count_present
    )
) != 0:

    raise ValueError(
        "OSQ020C routing does not match OSQ010C=1."
    )


print(
    "PASS: Spine-fracture count routing validated."
)


# ============================================================
# 20. CDC CHECK - OSQ020C
# ============================================================

check_frequency(
    "OSQ020C",
    {
        1: 67,
        2: 8,
        3: 2
    },
    3738
)


# ============================================================
# 21. GENERAL FRACTURE-LOOP FIELD VALIDATION
# ============================================================

#
# Numerous OSD030 / OSQ040 / OSD050 variables represent
# repeated hip, wrist, and spine fracture events.
#
# The CDC also notes that age values for participants
# 80+ are coded as 80 for disclosure protection.
#
# Rather than modifying any of these values, validate
# that every populated field remains whole-number.
#

FRACTURE_LOOP_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if (
        col.startswith("OSD030")
        or
        col.startswith("OSQ040")
        or
        col.startswith("OSD050")
    )
]


print(
    "\n--- FRACTURE LOOP FIELD CHECK ---"
)

print(
    "Fracture-loop fields:",
    len(FRACTURE_LOOP_COLUMNS)
)


for col in FRACTURE_LOOP_COLUMNS:

    values = (
        df[col]
        .dropna()
        .astype("int64")
    )


    #
    # We don't rewrite values.
    # This is simply a structural check.
    #

    print(
        f"{col}: "
        f"nonmissing={len(values):,}"
    )


print(
    "PASS: All released fracture-loop fields "
    "remain intact."
)


# ============================================================
# 22. CDC CHECK - OSQ080
# ============================================================

#
# Doctor ever told participant about another fracture
# after age 20.
#

check_frequency(
    "OSQ080",
    {
        1: 922,
        2: 2890,
        9: 3
    },
    0
)


# ============================================================
# 23. OTHER-FRACTURE LOOP STRUCTURE
# ============================================================

#
# OSQ090A-H / OSQ100 / OSD110 / OSQ120 form the
# repeated "other bone fracture" loop.
#
# We leave every value exactly as released.
#

OTHER_FRACTURE_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if (
        col.startswith("OSQ090")
        or
        col.startswith("OSQ100")
        or
        col.startswith("OSD110")
        or
        col.startswith("OSQ120")
    )
]


print(
    "\n--- OTHER FRACTURE LOOP ---"
)

print(
    "Released loop fields:",
    len(OTHER_FRACTURE_COLUMNS)
)


for col in OTHER_FRACTURE_COLUMNS:

    print(
        f"{col}: "
        f"nonmissing="
        f"{int(df[col].notna().sum()):,}"
    )


print(
    "PASS: Other-fracture loop fields "
    "retained without modification."
)


# ============================================================
# 24. CDC CHECK - OSQ060
# ============================================================

#
# Ever told had osteoporosis / brittle bones.
#

check_frequency(
    "OSQ060",
    {
        1: 322,
        2: 3481,
        9: 12
    },
    0
)


# ============================================================
# 25. CDC CHECK - OSQ072
# ============================================================

#
# Doctor ever prescribed medicine for osteoporosis.
#
# Asked when:
#
#     OSQ060 = 1
#

check_frequency(
    "OSQ072",
    {
        1: 176,
        2: 141,
        9: 5
    },
    3493
)


osq060_yes = (
    df["OSQ060"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


osq072_present = (
    df["OSQ072"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        osq060_yes
        !=
        osq072_present
    )
) != 0:

    raise ValueError(
        "OSQ072 routing does not match OSQ060=1."
    )


print(
    "PASS: Osteoporosis-treatment routing validated."
)


# ============================================================
# 26. CDC CHECK - OSQ130
# ============================================================

#
# Ever taken prednisone/cortisone nearly every day
# for one month or longer.
#

check_frequency(
    "OSQ130",
    {
        1: 220,
        2: 3573,
        9: 22
    },
    0
)


# ============================================================
# 27. CDC CHECK - OSQ140Q
# ============================================================

#
# Duration of prednisone/cortisone use.
#
# CDC:
#
# 1-35 = 207
# 999  = 13
# Missing = 3595
#

check_range_special(
    "OSQ140Q",
    low=1,
    high=35,
    expected_range_count=207,
    expected_missing=3595,
    special_counts={
        999: 13
    }
)


# ============================================================
# 28. OSQ140Q ROUTING
# ============================================================

osq130_yes = (
    df["OSQ130"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


osq140q_present = (
    df["OSQ140Q"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        osq130_yes
        !=
        osq140q_present
    )
) != 0:

    raise ValueError(
        "OSQ140Q routing does not match OSQ130=1."
    )


print(
    "PASS: Steroid duration routing validated."
)


# ============================================================
# 29. CDC CHECK - OSQ140U
# ============================================================

#
# Duration unit:
#
# 1 = Month
# 2 = Year
#
# Unit is populated only for ordinary 1-35
# OSQ140Q values, not Don't Know 999.
#

check_frequency(
    "OSQ140U",
    {
        1: 133,
        2: 74
    },
    3608
)


regular_osq140q = (
    df["OSQ140Q"]
    .between(
        1,
        35
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


osq140u_present = (
    df["OSQ140U"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        regular_osq140q
        !=
        osq140u_present
    )
) != 0:

    raise ValueError(
        "OSQ140U routing does not match "
        "ordinary OSQ140Q values."
    )


print(
    "PASS: OSQ140Q/OSQ140U routing validated."
)


# ============================================================
# 30. CDC CHECK - OSQ150
# ============================================================

#
# Either biological parent ever diagnosed
# with osteoporosis.
#

check_frequency(
    "OSQ150",
    {
        1: 458,
        2: 3138,
        7: 3,
        9: 216
    },
    0
)


# ============================================================
# 31. CDC CHECK - OSQ160A / OSQ160B
# ============================================================

#
# Check-all-that-apply fields.
#

check_frequency(
    "OSQ160A",
    {
        1: 436
    },
    3379
)


check_frequency(
    "OSQ160B",
    {
        2: 30
    },
    3785
)


# ============================================================
# 32. CDC CHECK - OSQ170
# ============================================================

#
# Biological mother ever fractured hip.
#

check_frequency(
    "OSQ170",
    {
        1: 256,
        2: 3416,
        9: 143
    },
    0
)


# ============================================================
# 33. CDC CHECK - OSQ180
# ============================================================

#
# Mother's age at first hip fracture.
#
# CDC:
#
# 19-100 = 242
# 999    = 14
# Missing = 3559
#

check_range_special(
    "OSQ180",
    low=19,
    high=100,
    expected_range_count=242,
    expected_missing=3559,
    special_counts={
        999: 14
    }
)


# ============================================================
# 34. OSQ170 / OSQ180 ROUTING
# ============================================================

mother_fracture_yes = (
    df["OSQ170"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


osq180_present = (
    df["OSQ180"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        mother_fracture_yes
        !=
        osq180_present
    )
) != 0:

    raise ValueError(
        "OSQ180 routing does not match OSQ170=1."
    )


print(
    "PASS: Mother's fracture-age routing validated."
)


# ============================================================
# 35. CDC CHECK - OSQ190
# ============================================================

#
# Asked when mother's exact age is unknown.
#
# Attached:
#
# 1 Under 50 = 3
# 2 50+      = 10
# 9 Don't know = 1
#

check_frequency(
    "OSQ190",
    {
        1: 3,
        2: 10,
        9: 1
    },
    3801
)


mother_age_unknown = (
    df["OSQ180"]
    .eq(999)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


osq190_present = (
    df["OSQ190"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        mother_age_unknown
        !=
        osq190_present
    )
) != 0:

    raise ValueError(
        "OSQ190 routing does not match OSQ180=999."
    )


print(
    "PASS: Mother's under/over-50 routing validated."
)


# ============================================================
# 36. CDC CHECK - OSQ200
# ============================================================

#
# Biological father ever fractured hip.
#

check_frequency(
    "OSQ200",
    {
        1: 81,
        2: 3550,
        7: 1,
        9: 183
    },
    0
)


# ============================================================
# 37. CDC CHECK - OSQ210
# ============================================================

#
# Father's age at first hip fracture.
#
# CDC:
#
# 19-101 = 77
# 999    = 4
# Missing = 3734
#

check_range_special(
    "OSQ210",
    low=19,
    high=101,
    expected_range_count=77,
    expected_missing=3734,
    special_counts={
        999: 4
    }
)


# ============================================================
# 38. OSQ200 / OSQ210 ROUTING
# ============================================================

father_fracture_yes = (
    df["OSQ200"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


osq210_present = (
    df["OSQ210"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        father_fracture_yes
        !=
        osq210_present
    )
) != 0:

    raise ValueError(
        "OSQ210 routing does not match OSQ200=1."
    )


print(
    "PASS: Father's fracture-age routing validated."
)


# ============================================================
# 39. CDC CHECK - OSQ220
# ============================================================

#
# Asked when father's exact age is unknown.
#

check_frequency(
    "OSQ220",
    {
        1: 2,
        2: 2
    },
    3811
)


father_age_unknown = (
    df["OSQ210"]
    .eq(999)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


osq220_present = (
    df["OSQ220"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        father_age_unknown
        !=
        osq220_present
    )
) != 0:

    raise ValueError(
        "OSQ220 routing does not match OSQ210=999."
    )


print(
    "PASS: Father's under/over-50 routing validated."
)


# ============================================================
# 40. FINAL ZERO CHECK BEFORE EXPORT
# ============================================================

#
# Attached source contained no zeros.
#
# Int64 conversion must not create any zeros.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK ---"
)


print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zero values were created:\n"
        + str(
            final_zero_counts
        )
    )


print(
    "PASS: No zero values created."
)


# ============================================================
# 41. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# OSQ_H requires ZERO numeric corrections.
#
# Therefore every numeric value must remain identical
# to the original XPT.
#
# Int64 conversion changes formatting only:
#
#     73557.0 -> 73557
#     1.0     -> 1
#     999.0   -> 999
#
# Numeric meaning remains identical.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            fidelity_failures
        )
    )


print(
    "PASS: All 91 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 42. EXPORT CSV
# ============================================================

#
# Standard NHANES fidelity export:
#
# NO:
#
#     .round()
#
# NO:
#
#     float_format=
#
#
# OSQ_H itself contains no true decimals.
#
# We still keep the same export approach so your overall
# NHANES workflow remains safe for Dietary/Lab/Exam files
# where real decimal values must remain AS IS.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 43. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 44. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: All 91 CSV variables round-trip "
    "to exact source XPT numeric values."
)


# ============================================================
# 45. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 46. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 47. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(
            TINY_VALUE
        )
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 48. RAW CSV .0 CHECK
# ============================================================

#
# OSQ_H has NO genuine decimal variables.
#
# Therefore checking ALL 91 variables for ".0"
# is safe for THIS file.
#
#
# Examples:
#
# SEQN
#     73557.0 -> 73557
#
# OSQ010A
#     1.0 -> 1
#
# OSD030AA
#     65.0 -> 65
#
# OSQ140Q
#     12.0 -> 12
#
# OSQ180
#     999.0 -> 999
#
#
# DO NOT use this global rule for datasets containing
# genuine decimal values.
#

print(
    "\n--- RAW CSV .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in any of the 91 OSQ_H fields."
)


# ============================================================
# 49. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)


print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values found "
        "in final CSV:\n"
        + str(
            csv_zero_counts
        )
    )


print(
    "PASS: Final CSV contains no zero values."
)


# ============================================================
# 50. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 51. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83729:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 52. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "OSQ_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Questionnaire - Osteoporosis"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original ordinary zeros:",
    true_zero_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "String variables: NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "Hip fracture yes:",
    int(
        df["OSQ010A"]
        .eq(1)
        .sum()
    )
)


print(
    "Wrist fracture yes:",
    int(
        df["OSQ010B"]
        .eq(1)
        .sum()
    )
)


print(
    "Spine fracture yes:",
    int(
        df["OSQ010C"]
        .eq(1)
        .sum()
    )
)


print(
    "Other fracture yes:",
    int(
        df["OSQ080"]
        .eq(1)
        .sum()
    )
)


print(
    "Osteoporosis diagnosis yes:",
    int(
        df["OSQ060"]
        .eq(1)
        .sum()
    )
)


print(
    "Osteoporosis RX yes:",
    int(
        df["OSQ072"]
        .eq(1)
        .sum()
    )
)


print(
    "Prednisone/cortisone use yes:",
    int(
        df["OSQ130"]
        .eq(1)
        .sum()
    )
)


print(
    "Parent osteoporosis yes:",
    int(
        df["OSQ150"]
        .eq(1)
        .sum()
    )
)


print(
    "Mother fractured hip yes:",
    int(
        df["OSQ170"]
        .eq(1)
        .sum()
    )
)


print(
    "Father fractured hip yes:",
    int(
        df["OSQ200"]
        .eq(1)
        .sum()
    )
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0': "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 OSQ_H XPT -> CSV
Rows: 3,815
Columns: 91
PASS: Structure = 3,815 rows x 91 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
String fields: []
Whole-number/code fields: 91

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Ordinary numeric zeros: 0
PASS: Attached OSQ_H contains 0 tiny artifacts and 0 ordinary zeros.

PASS: No numeric correction required.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional variables across all 91 fields.

--- SEQN ---
Minimum: 73557
Maximum: 83729
Missing: 0
Duplicates: 0
PASS: SEQN 73557-83729 validated.

PASS: All 91 OSQ_H fields converted to nullable Int64 without rounding.

--- OSQ010A ---
Observed: {np.int64(1): 83, np.int64(2): 3731, np.int64(9): 1}
Missing: 0
PASS: OSQ010A matches expected CDC release frequencies.

--- OSQ010B ---
Observed: {np.int64(1): 316, np.int64(2): 3496, np.int64(9): 3}
Missing: 0
PASS: OSQ010B matches expected CDC release frequencies.

--- OSQ010C ---
Observed: {np.int64(1): 77, n